<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-09-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-100.

Características do dataset:

- 60.000 imagens RGB
- 100 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar100

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-100 utilizando `tensorflow.keras.datasets.cifar100.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

As imagens originais do CIFAR-100 possuem formato `(32, 32, 3)`: 32 pixels de altura, 32 de largura e 3 canais RGB. Após o flatten, cada imagem passa a ter `32 * 32 * 3 = 3072` features.

O flatten é necessário porque uma MLP recebe entradas tabulares/vetoriais no formato `(amostras, features)`, e não tensores de imagem com altura, largura e canais. A normalização é importante porque transforma os pixels da escala 0-255 para 0-1, tornando o treinamento por gradiente mais estável, reduzindo oscilações e facilitando a comparação entre hiperparâmetros.

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


SEED = 42
VALIDATION_SIZE = 0.20
DATA_SAMPLE_SIZE = 10000
MAX_ITER = 25
BATCH_SIZE = 256

mlflow.set_tracking_uri("file:./mlruns")
mlflow.set_experiment("assignment")


def load_data(seed=SEED, sample_size=DATA_SAMPLE_SIZE):
    from tensorflow.keras.datasets import cifar100

    (X_train_full, y_train_full), _ = cifar100.load_data(label_mode="fine")

    X = X_train_full.astype(np.float32) / 255.0
    y = y_train_full.ravel().astype(np.int64)

    X = X.reshape(X.shape[0], -1)

    if sample_size is not None and sample_size < X.shape[0]:
        _, X, _, y = train_test_split(
            X,
            y,
            test_size=sample_size,
            stratify=y,
            random_state=seed,
        )

    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        test_size=VALIDATION_SIZE,
        stratify=y,
        random_state=seed,
    )

    return X_train, X_val, y_train, y_val


X_train, X_val, y_train, y_val = load_data(SEED)

print("Treino:", X_train.shape)
print("Validação:", X_val.shape)
print("Features por imagem:", X_train.shape[1])
print("Classes no treino:", len(np.unique(y_train)))

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

Na primeira camada, considerando a arquitetura `(64,)`, existem `3072 * 64 + 64 = 196672` parâmetros, contando pesos e vieses. A ReLU retorna zero para entradas negativas e mantém valores positivos, permitindo não linearidade com menor saturação que `logistic` e `tanh`.

MLPs possuem muitos parâmetros ao trabalhar com imagens porque, após o flatten, cada pixel/canal vira uma feature conectada aos neurônios da próxima camada. Diferente de CNNs, a MLP não compartilha filtros nem explora localidade espacial, então o número de pesos cresce rapidamente.

In [ ]:
def train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        learning_rate_init=learning_rate,
        max_iter=max_iter,
        batch_size=batch_size,
        early_stopping=True,
        validation_fraction=0.10,
        n_iter_no_change=5,
        random_state=seed,
    )
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1_score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

A accuracy representa a proporção total de acertos do modelo. Precision mede, entre as amostras previstas como determinada classe, quantas realmente pertenciam a ela; recall mede, entre as amostras reais dessa classe, quantas foram encontradas pelo modelo. O f1-score é importante quando queremos equilibrar precision e recall em uma única métrica, especialmente em problemas multiclasse como CIFAR-100, nos quais o desempenho pode variar bastante entre classes.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

O MLflow foi usado para registrar hiperparâmetros, métricas e tempo de treinamento de cada execução. Esse rastreamento facilita comparar experimentos, identificar a melhor configuração e manter reprodutibilidade.

In [ ]:
experiment_results = []


def run_experiment(
    run_name,
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
    max_iter=MAX_ITER,
    batch_size=BATCH_SIZE,
):
    params = {
        "activation": activation,
        "hidden_layers": str(hidden_layers),
        "learning_rate": learning_rate,
        "max_iter": max_iter,
        "batch_size": batch_size,
        "seed": SEED,
        "sample_size": DATA_SAMPLE_SIZE,
    }

    start = time.time()
    with mlflow.start_run(run_name=run_name):
        for key, value in params.items():
            mlflow.log_param(key, value)

        model = train_mlp(
            X_train,
            y_train,
            activation=activation,
            hidden_layers=hidden_layers,
            learning_rate=learning_rate,
            seed=SEED,
            max_iter=max_iter,
            batch_size=batch_size,
        )

        training_time = time.time() - start
        metrics = evaluate(model, X_val, y_val)
        train_accuracy = accuracy_score(y_train, model.predict(X_train))

        logged_metrics = {
            **metrics,
            "training_time": training_time,
            "train_accuracy": train_accuracy,
            "loss": model.loss_,
            "n_iter": model.n_iter_,
        }

        for key, value in logged_metrics.items():
            mlflow.log_metric(key, value)

    row = {
        "run_name": run_name,
        **params,
        **logged_metrics,
    }
    experiment_results.append(row)
    return row, model


baseline_result, baseline_model = run_experiment(
    "baseline_relu_64_lr_0.001",
    activation="relu",
    hidden_layers=(64,),
    learning_rate=0.001,
)

pd.DataFrame([baseline_result])

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

As funções de ativação foram comparadas mantendo a mesma arquitetura, learning rate e seed. A ReLU costuma ser amplamente usada em Deep Learning porque é simples, reduz saturação e facilita o fluxo do gradiente em redes maiores.

In [ ]:
activation_results = []

for activation in ["logistic", "tanh", "relu"]:
    result, _ = run_experiment(
        run_name=f"activation_{activation}",
        activation=activation,
        hidden_layers=(64,),
        learning_rate=0.001,
    )
    activation_results.append(result)

activation_df = pd.DataFrame(activation_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
activation_df

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

Redes maiores aumentam a capacidade de representação, mas também aumentam custo computacional e risco de overfitting. Por isso a análise considera desempenho de validação, tempo de treinamento, loss e diferença entre acurácia de treino e validação.

In [ ]:
architecture_results = []

for hidden_layers in [(32,), (64,), (128, 64), (256, 128)]:
    result, _ = run_experiment(
        run_name=f"architecture_{hidden_layers}",
        activation="relu",
        hidden_layers=hidden_layers,
        learning_rate=0.001,
    )
    architecture_results.append(result)

architecture_df = pd.DataFrame(architecture_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
architecture_df

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
learning_rate_results = []

for learning_rate in [0.1, 0.01, 0.001]:
    result, _ = run_experiment(
        run_name=f"learning_rate_{learning_rate}",
        activation="relu",
        hidden_layers=(64,),
        learning_rate=learning_rate,
    )
    learning_rate_results.append(result)

learning_rate_df = pd.DataFrame(learning_rate_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
learning_rate_df

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

## Discussão final

O comportamento da loss depende diretamente do learning rate. Valores altos, como `0.1`, podem gerar oscilações e dificultar a convergência; valores menores, como `0.001`, tendem a produzir treinamento mais estável, ainda que possam exigir mais iterações.

A arquitetura influencia desempenho e custo. Redes maiores têm mais capacidade, mas não garantem melhora contínua e podem apresentar overfitting quando a acurácia de treino fica bem acima da validação. A ativação também altera a convergência: `logistic` tende a saturar mais, `tanh` costuma ser mais estável que `logistic`, e `relu` geralmente converge melhor em redes profundas.

A melhor configuração final deve ser escolhida pela maior combinação de `f1_score` e `accuracy` na validação, considerando também `training_time`, `loss` e a diferença entre treino e validação. As principais dificuldades foram trabalhar com 100 classes, controlar custo computacional e comparar hiperparâmetros de forma justa.

MLPs possuem limitações para imagens porque ignoram a estrutura espacial: após o flatten, relações locais entre pixels vizinhos são perdidas. CNNs são mais adequadas porque usam convoluções, filtros locais e compartilhamento de pesos.

O backpropagation contribui para o aprendizado calculando o efeito de cada peso no erro final e propagando esse erro de volta pela rede. Com isso, o otimizador ajusta os pesos para reduzir a loss ao longo das iterações.

### Interpretações dos experimentos

Na Questão 4, o melhor experimento é aquele com maior `f1_score` e `accuracy` na validação; a configuração mais estável é a que combina boa métrica, menor loss e menor diferença entre `train_accuracy` e validação. Na Questão 5, a melhor ativação deve ser lida em `activation_df`; normalmente `relu` oferece melhor convergência prática. Na Questão 6, redes maiores não melhoram sempre: `(256, 128)` tem maior custo, enquanto arquiteturas intermediárias podem ter melhor tradeoff. Na Questão 7, o melhor learning rate deve ser lido em `learning_rate_df`; em geral `0.001` é mais estável e `0.1` é o mais propenso a instabilidade.

In [ ]:
results_df = pd.DataFrame(experiment_results).sort_values(
    ["f1_score", "accuracy"],
    ascending=False,
)
results_df